## LiveCodeBench

| Release | 기간                |      문제 수 |
| ------- | ----------------- | --------: |
| v1      | 2023.05 ~ 2024.03 |       400 |
| v2      | ~ 2024.05         |       511 |
| v3      | ~ 2024.07         |       612 |
| v4      | ~ 2024.09         |       713 |
| v5      | ~ 2025.01         |       880 |
| **v6**  | ~ **2025.04**     | **1,055** |

- [각 release에는 이전 release의 문제 + 새로 추가된 문제로 누적하여 구성된다.](https://huggingface.co/datasets/livecodebench/code_generation_lite/blob/main/code_generation_lite.py?utm_source=chatgpt.com)

- LiveCodeBench-v6(1055 prolbems) : v5(880) + new(175)
    - The difficulty distribution : 75 Easy / 75 Medium / 25 Hard

reference : https://www.emergentmind.com/topics/livecodebench-v5-v6-pro   
hagginface : livecodebench/code_generation_lite, release_v6, split : test(train/val없음)

### 1. Dataset 기본 구조 확인

In [1]:
from datasets import load_dataset

HF_CACHE = "/mnt/hdd/hf_cache"

dataset = load_dataset(
    "livecodebench/code_generation_lite",
    version_tag="release_v6",
    cache_dir=HF_CACHE,
)

lcb_v6 = dataset["test"]

print("=" * 60)
print("LiveCodeBench v6 Dataset")
print("=" * 60)

print(f"Number of samples : {len(lcb_v6)}")
print(f"Columns           : {lcb_v6.column_names}")

/mnt/hdd/conda_envs/slm/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LiveCodeBench v6 Dataset
Number of samples : 1055
Columns           : ['question_title', 'question_content', 'platform', 'question_id', 'contest_id', 'contest_date', 'starter_code', 'difficulty', 'public_test_cases', 'private_test_cases', 'metadata']


| 컬럼                   | 설명                        | 비고                                                                                     |
| -------------------- | ------------------------- | -------------------------------------------------------------------------------------- |
| `question_title`     | 문제의 제목                    | 문제를 식별하거나 사람이 문제 내용을 빠르게 파악할 때 사용                                                      |
| `question_content`   | 문제의 전체 설명 및 문제 해결에 필요한 내용 | **핵심 컬럼.** 모델에게 실제 문제로 제공되는 내용이며, planning/code generation 실험의 입력이 됨                   |
| `platform`           | 문제가 출제된 플랫폼               | `atcoder`, `leetcode`, `codeforces`                                                    |
| `question_id`        | 각 문제의 고유 식별자              | **중요.** 문제 중복 확인, 결과 매칭, 실험 재현 등에 사용                                                   |
| `contest_id`         | 문제가 속한 contest의 식별자       | 별도의 contest 데이터가 있는 것이 아니라, 각 문제에 출제 contest ID가 기록된 것                                 |
| `contest_date`       | 해당 문제가 출제/공개된 날짜          | **Temporal benchmark 특성 때문에 중요.** 데이터의 시간적 분포 및 contamination 분석에 활용 가능                |
| `starter_code`       | 문제에서 제공하는 시작 코드/코드 템플릿    | 특히 LeetCode 형태의 문제에서 의미가 있음. 코드 생성 입력 구성에 영향을 줄 수 있음                                   |
| `difficulty`         | 문제 난이도                    | 현재 v6에서는 `easy`, `medium`, `hard`.                      |
| `public_test_cases`  | 공개 테스트 케이스                | 모델이 문제를 해결할 때 참고할 수 있는 테스트 입력/출력 정보                                                    |
| `private_test_cases` | 비공개 테스트 케이스               | 평가에 사용되는 hidden/private 테스트 정보. **실제 정답 코드의 일반화/통과 여부 평가에 중요**                         |
| `metadata`           | 문제와 관련된 추가 메타데이터          | v6에서는 플랫폼에 따라 다름. 확인한 결과 **AtCoder/Codeforces는 `{}`, LeetCode는 주로 `func_name`** 형태 |


### 2. Difficulty 분석

In [2]:
from collections import Counter

print("\n" + "=" * 60)
print("Difficulty Distribution")
print("=" * 60)

difficulty_counts = Counter(lcb_v6["difficulty"])

for difficulty, count in difficulty_counts.items():
    ratio = count / len(lcb_v6) * 100
    print(f"{difficulty:10s}: {count:4d} ({ratio:5.1f}%)")


Difficulty Distribution
easy      :  322 ( 30.5%)
medium    :  383 ( 36.3%)
hard      :  350 ( 33.2%)


### 3. Platform 분석

findings : codeforces플랫폼의 데이터셋 수가 매우 적음

In [3]:
print("\n" + "=" * 60)
print("Platform Distribution")
print("=" * 60)

platform_counts = Counter(lcb_v6["platform"])

for platform, count in platform_counts.most_common():
    ratio = count / len(lcb_v6) * 100
    print(f"{str(platform):15s}: {count:4d} ({ratio:5.1f}%)")


Platform Distribution
atcoder        :  602 ( 57.1%)
leetcode       :  444 ( 42.1%)
codeforces     :    9 (  0.9%)


### 4. Metadata 분석

finds : metadata column은 주로 LeetCode 함수 시그니처/함수명 정보에 대한 것.

In [4]:
# import pprint

# print("\n" + "=" * 60)
# print("Metadata Examples")
# print("=" * 60)

# for i in range(5):
#     print(f"\n--- Sample {i} ---")
#     print("Title     :", lcb_v6[i]["question_title"])
#     print("Platform  :", lcb_v6[i]["platform"])
#     print("Difficulty:", lcb_v6[i]["difficulty"])
#     print("Metadata  :")
#     pprint.pp(lcb_v6[i]["metadata"])

In [5]:
# from collections import Counter

# print("\n" + "=" * 60)
# print("Metadata Availability")
# print("=" * 60)

# metadata_counts = Counter(
#     str(sample["metadata"])
#     for sample in lcb_v6
# )

# print(f"Unique metadata values: {len(metadata_counts)}")

# for metadata, count in metadata_counts.most_common(20):
#     print(f"{count:4d} : {metadata}")

In [6]:
from collections import Counter

print("\n" + "=" * 60)
print("Platform × Metadata Availability")
print("=" * 60)

for platform in sorted(set(lcb_v6["platform"])):
    samples = [
        x for x in lcb_v6
        if x["platform"] == platform
    ]

    empty = sum(
        1 for x in samples
        if x["metadata"] == "{}"
    )

    non_empty = len(samples) - empty

    print(f"\n{platform}")
    print(f"  Total     : {len(samples)}")
    print(f"  Empty     : {empty}")
    print(f"  Non-empty : {non_empty}")


Platform × Metadata Availability

atcoder
  Total     : 602
  Empty     : 602
  Non-empty : 0

codeforces
  Total     : 9
  Empty     : 9
  Non-empty : 0

leetcode
  Total     : 444
  Empty     : 0
  Non-empty : 444


### 5. Contest / 날짜 분포

findins : AtCoder와 LeetCode는 2025년 4월까지 지속적으로 수집되는 반면, Codeforces는 2개월 동안의 수집으로 9문제 업데이트 됨.

In [7]:
print("\n" + "=" * 60)
print("Platform × Date Range")
print("=" * 60)

for platform in sorted(set(lcb_v6["platform"])):
    dates = [
        x["contest_date"]
        for x in lcb_v6
        if x["platform"] == platform
    ]

    print(f"\n{platform}")
    print(f"  Earliest: {min(dates)}")
    print(f"  Latest  : {max(dates)}")


Platform × Date Range

atcoder
  Earliest: 2023-05-13T00:00:00
  Latest  : 2025-04-06T00:00:00

codeforces
  Earliest: 2023-08-21T00:00:00
  Latest  : 2023-10-17T00:00:00

leetcode
  Earliest: 2023-05-07T00:00:00
  Latest  : 2025-04-05T19:30:00


### 6. Question ID 중복 확인

In [8]:
from collections import Counter

print("\n" + "=" * 60)
print("Question ID")
print("=" * 60)

question_ids = lcb_v6["question_id"]
id_counts = Counter(question_ids)

duplicates = {
    qid: count
    for qid, count in id_counts.items()
    if count > 1
}

print(f"Total IDs      : {len(question_ids)}")
print(f"Unique IDs     : {len(id_counts)}")
print(f"Duplicate IDs  : {len(duplicates)}")

if duplicates:
    print("\nDuplicates:")
    for qid, count in list(duplicates.items())[:20]:
        print(f"  {qid}: {count}")


Question ID
Total IDs      : 1055
Unique IDs     : 1055
Duplicate IDs  : 0


In [9]:
print("\n" + "=" * 60)
print("Contest Statistics")
print("=" * 60)

contest_counts = Counter(lcb_v6["contest_id"])

print(f"Unique contests: {len(contest_counts)}")

print("\nLargest contests:")
for contest_id, count in contest_counts.most_common(20):
    print(f"{contest_id}: {count}")


Contest Statistics
Unique contests: 267

Largest contests:
abc366: 7
abc367: 7
abc368: 7
abc370: 7
abc371: 7
abc374: 7
abc375: 7
abc376: 7
abc377: 7
abc378: 7
abc379: 7
abc380: 7
abc384: 7
abc388: 7
abc390: 7
abc394: 7
abc396: 7
abc397: 7
abc301: 6
abc302: 6
